# PAYMENTS CLEANING

In [ ]:
import pandas as pd
payments = pd.read_csv("../Data/sanlam_raw_payments.csv")

payments.head()

In [ ]:
# Check to see if the transaction date has any NA values 
payments['transaction_date'].isna().sum()


In [ ]:
# Step 1: Create a parser for the known formats
def parse_transaction_date(date_str):
    if pd.isna(date_str):
        return pd.NaT

    date_str = str(date_str).strip()

    try:
        # DD/MM/YYYY
        if '/' in date_str and len(date_str.split('/')[0]) == 2:
            return pd.to_datetime(date_str, format='%d/%m/%Y')

        # MM-DD-YYYY
        elif '-' in date_str and len(date_str.split('-')[0]) == 2:
            return pd.to_datetime(date_str, format='%m-%d-%Y')

        # YYYY-MM-DD
        elif len(date_str.split('-')[0]) == 4:
            return pd.to_datetime(date_str, format='%Y-%m-%d')

        # YYYY/MM/DD
        elif len(date_str.split('/')[0]) == 4:
            return pd.to_datetime(date_str, format='%Y/%m/%d')

        return pd.NaT

    except:
        return pd.NaT

In [ ]:
# Step 2: Parse the dates
payments['parse_transaction_date'] = (
    payments['transaction_date']
    .apply(parse_transaction_date)
)

In [ ]:
payments.head()

In [ ]:
# Step 3: Find malformed dates
bad_dates = payments[
    payments['transaction_date'].notna() &
    payments['parse_transaction_date'].isna()
]

In [ ]:
# Step 4: Count malformed dates
bad_dates.shape[0]

In [ ]:
#Step 5
payments['transaction_date_std'] = (
    payments['parse_transaction_date']
    .dt.strftime('%d-%m-%Y')
)

In [ ]:
payments.columns.tolist()

In [ ]:
payments.head(30)

In [ ]:
#Rename the transaction_date_std to transaction_date_new
payments.rename(columns={'transaction_date_std': 'transaction_date_new'}, inplace=True)

In [ ]:
# Remove the parse_transaction_date coloumn 
payments.drop(columns=['parse_transaction_date'], inplace=True)

In [ ]:
# Remove the transaction_date coloumn (original)
payments.drop(columns=['transaction_date'], inplace=True)

In [ ]:
cols = payments.columns.tolist()

# Remove the column from its current position
cols.remove('transaction_date_new')

# Insert it into the 4th position
cols.insert(3, 'transaction_date_new')

# Reorder the dataframe
payments = payments[cols]

In [ ]:
payments.dtypes

In [ ]:
# Convert string to datetime 
payments['transaction_date_new'] = pd.to_datetime(
    payments['transaction_date_new'],
    format='%d-%m-%Y',
    errors='coerce'
)

In [ ]:
## Check the malformed dates (QUARTINIE ALL 616 ROWS)
payments['transaction_date_new'].isna().sum()

In [ ]:
# Check the currency distinct values 
payments['currency'].unique().tolist()

In [ ]:
payments['currency'] = payments['currency'].replace('R', 'ZAR')

In [ ]:
# Check the distinct values for payment method. 
payments['payment_method'].unique().tolist()

In [ ]:
payments['payment_method'].isna().sum()

In [ ]:
# Set 1202payment_method0 payment method to "Unknown"
payments['payment_method'] = payments['payment_method'].fillna('Unknown')

In [ ]:
payments['payment_method'].head(10)

In [ ]:
payments['payment_method'].unique().tolist()

In [ ]:
#Remove trailling spaces and whitespaces. 
payments['payment_method'] = payments['payment_method'].str.strip()

In [ ]:
payments.head(10)

In [ ]:
# Check if the coloumns contain R or ZAR 
payments[
    payments['amount'].astype(str).str.contains('R|ZAR', na=False)
]

In [ ]:
#Replace the R and ZAR from amounts 
payments['amount'] = payments['amount'].str.replace('ZAR', '', regex=False)
payments['amount'] = payments['amount'].str.replace('R', '', regex=False)
payments['amount'] = payments['amount'].str.strip()

In [ ]:
#Check the distinct values in transcation type
payments['transaction_type'].unique().tolist()

In [82]:
#Display the value counts for each transaction type
payments['payment_status'].value_counts(dropna=False)

payment_status
Successful    30119
Reversed      10020
Failed         9991
Pending        9870
Name: count, dtype: int64

In [ ]:
# Merge success into successful. 
payments['payment_status'] = payments['payment_status'].replace({
    'success': 'Successful'
})

In [81]:
# Merge paid into Successful
payments['payment_status'] = payments['payment_status'].replace({
    'Paid': 'Successful'
})

In [68]:
payments['transaction_type'] = payments['transaction_type'].str.strip()

In [74]:
payments.columns

Index(['source_row_id', 'transaction_ref', 'policy_id', 'transaction_date_new',
       'client_id', 'amount', 'transaction_type', 'currency', 'payment_method',
       'payment_status', 'source_system'],
      dtype='str')

In [78]:
#Create a status crosswalk. 
status_crosswalk = {
    'success': 'Successful',
    'Successful': 'Successful',
    'Paid': 'Paid',
    'Reversed': 'Reversed',
    'Failed': 'Failed',
    'Pending': 'Pending'
}

In [79]:
payments['payment_status'] = (
    payments['payment_status']
    .map(status_crosswalk)
)

In [92]:
crosswalk = pd.DataFrame({
    'original_value': ['success', 'Successful', 'Paid', 'Reversed', 'Failed', 'Pending'],
    'standardized_value': ['Successful', 'Successful', 'Paid', 'Reversed', 'Failed', 'Pending']
})

print(crosswalk)

  original_value standardized_value
0        success         Successful
1     Successful         Successful
2           Paid               Paid
3       Reversed           Reversed
4         Failed             Failed
5        Pending            Pending


In [83]:
#Create a backup if anything goes wrong 
payments_backup = payments.copy()

In [85]:
payments.head(10)

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system
0,SPAY-0000000001,TX-S-0000000001,POL000003392,2022-11-15,CL00007794,"1,826.09",Commission,ZAR,Unknown,Reversed,LegacyLedger
1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,"-7,370.85",Refund,ZAR,EFT,Successful,PaymentsHub
2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,"4,318.19",EFT,ZAR,Card,Successful,LegacyLedger
3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,"4,114.14",Debit Order,ZAR,EFT,Successful,PaymentsHub
4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,"21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed
5,SPAY-0000000006,TX-S-0000000006,POL000008647,2025-03-09,CL00000156,NaN,Premium,USD,Unknown,Failed,BankFeed
6,SPAY-0000000007,TX-S-0000000007,POL000000436,2024-06-23,CL00003805,"(-24,653.54)",Reversal,ZAR,Cash,Successful,PaymentsHub
7,SPAY-0000000008,TX-S-0000000008,POL000024244,2022-05-31,CL00000252,5285.68,Premium,ZAR,Cash,Successful,PaymentsHub
8,SPAY-0000000009,TX-S-0000000009,POL000012945,2021-03-31,CL00007328,"20,646.63",Premium,ZAR,Debit Order,Reversed,LegacyLedger
9,SPAY-0000000010,TX-S-0000000010,POL000010985,2025-10-10,CL00004931,"-21,763.22",Commission Reversal,ZAR,Card,Successful,BankFeed


In [86]:
#Counts if the transcation ref appears more than once 
payments['transaction_ref'].value_counts()[payments['transaction_ref'].value_counts() > 1]

transaction_ref
TX-S-0000003579    4
TX-S-0000000069    3
TX-S-0000000072    3
TX-S-0000000347    3
TX-S-0000000441    3
                  ..
TX-S-0000051281    2
TX-S-0000051967    2
TX-S-0000052514    2
TX-S-0000053824    2
TX-S-0000056554    2
Name: count, Length: 696, dtype: int64

In [87]:
payments.loc[
    payments['transaction_ref'] == 'TX-S-0000003579'
]

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system
3578,SPAY-0000003579,TX-S-0000003579,POL000002804,2024-12-26,CL00005274,"-2.581,54",Commission Reversal,ZAR,Debit Order,Successful,LegacyLedger
8680,SPAY-0000008681,TX-S-0000003579,POL000025057,2020-06-21,CL00010811,"7,716.14",Commission,ZAR,Unknown,Successful,BankFeed
17171,SPAY-0000017172,TX-S-0000003579,POL000019398,2022-02-23,CL00009971,"-1,030.31",Refund,USD,Cash,Successful,LegacyLedger
20707,SPAY-0000020708,TX-S-0000003579,POL000023129,2025-06-01,CL00000029,"22,523.73",Commission,ZAR,Card,Pending,LegacyLedger


In [89]:
#Identify transactions linked to multiple clients 
client_counts = (
    payments.groupby('transaction_ref')['client_id'].nunique()
)

#References asscoiated with multiple clients 
conflicting_refs = client_counts[client_counts > 1].index

#Extract all affected records
quarantine_candindates = payments[payments['transaction_ref'].isin(conflicting_refs)].copy()

#Display the records for invesitagtion 
print(quarantine_candindates.sort_values('transaction_ref'))
print("Affected records:", len(quarantine_candindates))
print("Conflicting transaction references:", len(conflicting_refs))

         source_row_id  transaction_ref     policy_id transaction_date_new  \
11     SPAY-0000000012  TX-S-0000000012  POL000019827           2020-06-12   
296    SPAY-0000000297  TX-S-0000000012  POL000001610           2021-01-08   
61     SPAY-0000000062  TX-S-0000000062  POL000023466           2020-01-16   
92     SPAY-0000000093  TX-S-0000000062  POL000020505           2025-03-28   
67     SPAY-0000000068  TX-S-0000000068  POL000006536           2022-12-08   
...                ...              ...           ...                  ...   
58035  SPAY-0000058036  TX-S-0000052514  POL000016777           2020-10-15   
53823  SPAY-0000053824  TX-S-0000053824  POL000003554           2024-06-17   
54630  SPAY-0000054631  TX-S-0000053824  POL000013213           2022-01-13   
56553  SPAY-0000056554  TX-S-0000056554  POL000005682           2020-11-12   
58376  SPAY-0000058377  TX-S-0000056554  POL000017531           2021-08-09   

        client_id     amount transaction_type currency payment_

In [90]:
duplicate_clients = (
    payments.groupby('transaction_ref')['client_id']
    .nunique()
    .sort_values(ascending=False)
)

duplicate_clients[duplicate_clients > 1]

transaction_ref
TX-S-0000003579    4
TX-S-0000000902    3
TX-S-0000020581    3
TX-S-0000000072    3
TX-S-0000000069    3
                  ..
TX-S-0000051281    2
TX-S-0000051967    2
TX-S-0000052514    2
TX-S-0000053824    2
TX-S-0000056554    2
Name: client_id, Length: 696, dtype: int64

In [91]:
duplicate_policies = (
    payments.groupby('transaction_ref')['policy_id']
    .nunique()
)

duplicate_policies[duplicate_policies > 1]

transaction_ref
TX-S-0000000012    2
TX-S-0000000062    2
TX-S-0000000068    2
TX-S-0000000069    3
TX-S-0000000071    2
                  ..
TX-S-0000051281    2
TX-S-0000051967    2
TX-S-0000052514    2
TX-S-0000053824    2
TX-S-0000056554    2
Name: policy_id, Length: 696, dtype: int64